# 03 — Sovereign Stress DSA

This notebook extends the validated baseline stochastic DSA with **endogenous sovereign-risk feedback**.

Notebook 02 answered:

> Given the central fiscal path, what range of outcomes results from ordinary historically calibrated macro-financial uncertainty?

Notebook 03 asks:

> What happens when fiscal deterioration itself widens the French sovereign spread, raising refinancing costs and feeding back into debt?

The first objective is to calibrate the sovereign-spread reaction empirically from a euro-area country panel.

The proposed feedback equation is:

\[
s_{i,t}
=
\alpha_i
+
\gamma_t
+
\beta_d d_{i,t-1}
+
\beta_{pb} pb_{i,t}
+
\beta_g g_{i,t}
+
u_{i,t}
\]

where:

- \(s_{i,t}\): 10Y sovereign spread versus Germany;
- \(d_{i,t-1}\): lagged general-government debt/GDP;
- \(pb_{i,t}\): primary balance/GDP;
- \(g_{i,t}\): real GDP growth;
- \(\alpha_i\): country fixed effect;
- \(\gamma_t\): year fixed effect.

Once these coefficients have been inspected and validated, they will feed the recursive stress simulator.

In [ ]:
from pathlib import Path
from io import StringIO
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

try:
    import statsmodels.formula.api as smf
except ImportError as exc:
    raise ImportError(
        "Notebook 03 requires statsmodels. "
        "Install it with: pip install statsmodels"
    ) from exc


PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

STRESS_DIR = (
    PROCESSED_DIR
    / "sovereign_stress"
)

STRESS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

from src.models.stochastic_dsa import (
    build_market_curves_from_path,
    generate_calibrated_stochastic_path,
    percentile_frame,
    risk_probability_table,
    run_monte_carlo,
    simulate_stochastic_gg_dsa,
)

RANDOM_SEED = 42

print("Project root:", PROJECT_ROOT)

## 1. Load the Notebook 02 baseline Monte Carlo benchmark

Notebook 03 should not rerun the ordinary-times Monte Carlo merely to reproduce the benchmark. We load the compact outputs saved by Notebook 02.

In [ ]:
MONTE_CARLO_DIR = (
    PROCESSED_DIR
    / "monte_carlo"
)

MC_PATH = MONTE_CARLO_DIR / "monte_carlo_10000_paths.npz"
DEBT_FAN_PATH = MONTE_CARLO_DIR / "debt_fan_percentiles.csv"
INTEREST_FAN_PATH = MONTE_CARLO_DIR / "interest_fan_percentiles.csv"
OAT_FAN_PATH = MONTE_CARLO_DIR / "oat_fan_percentiles.csv"
RISK_PATH = MONTE_CARLO_DIR / "risk_probabilities.csv"

required_files = [
    MC_PATH,
    DEBT_FAN_PATH,
    INTEREST_FAN_PATH,
    OAT_FAN_PATH,
    RISK_PATH,
]

missing_files = [
    path
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Notebook 02 Monte Carlo outputs are missing:\n"
        + "\n".join(
            str(path)
            for path in missing_files
        )
    )

mc02 = np.load(MC_PATH)

debt_fan_02 = pd.read_csv(DEBT_FAN_PATH)
interest_fan_02 = pd.read_csv(INTEREST_FAN_PATH)
oat_fan_02 = pd.read_csv(OAT_FAN_PATH)
risk_02 = pd.read_csv(RISK_PATH)

print(
    "Notebook 02 path matrix:",
    mc02["debt_gdp"].shape,
)

risk_02[
    [
        "metric",
        "probability_pct",
    ]
]

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.fill_between(
    debt_fan_02["year"],
    debt_fan_02["p10"] * 100,
    debt_fan_02["p90"] * 100,
    alpha=0.15,
    label="Notebook 02: P10–P90",
)

ax.fill_between(
    debt_fan_02["year"],
    debt_fan_02["p25"] * 100,
    debt_fan_02["p75"] * 100,
    alpha=0.25,
    label="Notebook 02: P25–P75",
)

ax.plot(
    debt_fan_02["year"],
    debt_fan_02["p50"] * 100,
    linewidth=2,
    label="Notebook 02 median",
)

ax.set_title(
    "Baseline benchmark — ordinary-times Monte Carlo"
)

ax.set_xlabel("Year")
ax.set_ylabel("Debt / GDP (%)")
ax.grid(alpha=0.25)
ax.legend()

plt.show()

## 2. Euro-area sovereign panel

Germany is the benchmark used to construct spreads and is therefore not itself included as a dependent-variable observation.

The first panel uses France, Italy, Spain, Belgium, Portugal, Austria, Netherlands, Finland, Ireland and Greece over 2000–2025 where data permit.

In [ ]:
COUNTRIES = {
    "France": {
        "ecb": "FR",
        "eurostat": "FR",
        "ameco": "FRA",
    },
    "Italy": {
        "ecb": "IT",
        "eurostat": "IT",
        "ameco": "ITA",
    },
    "Spain": {
        "ecb": "ES",
        "eurostat": "ES",
        "ameco": "ESP",
    },
    "Belgium": {
        "ecb": "BE",
        "eurostat": "BE",
        "ameco": "BEL",
    },
    "Portugal": {
        "ecb": "PT",
        "eurostat": "PT",
        "ameco": "PRT",
    },
    "Austria": {
        "ecb": "AT",
        "eurostat": "AT",
        "ameco": "AUT",
    },
    "Netherlands": {
        "ecb": "NL",
        "eurostat": "NL",
        "ameco": "NLD",
    },
    "Finland": {
        "ecb": "FI",
        "eurostat": "FI",
        "ameco": "FIN",
    },
    "Ireland": {
        "ecb": "IE",
        "eurostat": "IE",
        "ameco": "IRL",
    },
    "Greece": {
        "ecb": "GR",
        "eurostat": "EL",
        "ameco": "GRC",
    },
}

GERMANY_ECB_CODE = "DE"

PANEL_START_YEAR = 2000
PANEL_END_YEAR = 2025

print("Panel countries:", len(COUNTRIES))

## 3. Cached public-data downloaders

The panel uses:

- ECB IRS harmonised monthly 10Y government yields;
- Eurostat `gov_10dd_edpt1` for Maastricht debt/GDP;
- Eurostat `nama_10_gdp` for real GDP growth;
- AMECO via DBnomics for primary balance/GDP.

All downloads are cached under `data/raw/sovereign_stress/`.

In [ ]:
STRESS_RAW_DIR = (
    RAW_DIR
    / "sovereign_stress"
)

ECB_CACHE_DIR = STRESS_RAW_DIR / "ecb"
EUROSTAT_CACHE_DIR = STRESS_RAW_DIR / "eurostat"
AMECO_CACHE_DIR = STRESS_RAW_DIR / "ameco"


def create_retry_session():
    retry_strategy = Retry(
        total=6,
        connect=6,
        read=6,
        status=6,
        backoff_factor=1.5,
        status_forcelist=[
            429,
            500,
            502,
            503,
            504,
        ],
        allowed_methods=frozenset(["GET"]),
        respect_retry_after_header=True,
    )

    adapter = HTTPAdapter(
        max_retries=retry_strategy
    )

    session = requests.Session()
    session.mount("https://", adapter)

    session.headers.update({
        "User-Agent":
            "french-oat-model/1.0",
    })

    return session


HTTP_SESSION = create_retry_session()

In [ ]:
def fetch_ecb_10y(
    country_code,
    start_period="2000-01",
    end_period="2025-12",
    force_refresh=False,
):
    """
    Fetch the ECB monthly harmonised 10Y long-term rate.
    """

    ECB_CACHE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    cache_path = (
        ECB_CACHE_DIR
        / (
            f"ecb_10y_{country_code}_"
            f"{start_period}_{end_period}.csv"
        )
    )

    if (
        cache_path.exists()
        and not force_refresh
    ):
        return pd.read_csv(
            cache_path,
            parse_dates=["TIME_PERIOD"],
        )

    series_key = (
        f"M.{country_code}."
        "L.L40.CI.0000.EUR.N.Z"
    )

    url = (
        "https://data-api.ecb.europa.eu/"
        f"service/data/IRS/{series_key}"
    )

    response = HTTP_SESSION.get(
        url,
        params={
            "startPeriod": start_period,
            "endPeriod": end_period,
            "format": "csvdata",
            "detail": "dataonly",
        },
        timeout=(10, 180),
    )

    response.raise_for_status()

    df = pd.read_csv(
        StringIO(response.text)
    )

    required = {
        "TIME_PERIOD",
        "OBS_VALUE",
    }

    missing = (
        required
        - set(df.columns)
    )

    if missing:
        raise ValueError(
            f"ECB response missing: {missing}"
        )

    df = df[
        [
            "TIME_PERIOD",
            "OBS_VALUE",
        ]
    ].copy()

    df["TIME_PERIOD"] = pd.to_datetime(
        df["TIME_PERIOD"],
        errors="coerce",
    )

    df["OBS_VALUE"] = pd.to_numeric(
        df["OBS_VALUE"],
        errors="coerce",
    )

    df = (
        df
        .dropna()
        .sort_values("TIME_PERIOD")
        .reset_index(drop=True)
    )

    df.to_csv(
        cache_path,
        index=False,
    )

    return df


def monthly_rate_to_annual(
    df,
    value_name,
):
    result = df.copy()

    result["year"] = (
        result["TIME_PERIOD"]
        .dt.year
    )

    result[value_name] = (
        result["OBS_VALUE"]
        / 100
    )

    return (
        result[
            [
                "year",
                value_name,
            ]
        ]
        .groupby(
            "year",
            as_index=False,
        )[value_name]
        .mean()
    )

In [ ]:
def _jsonstat_category_labels(
    category_index,
):
    if isinstance(
        category_index,
        dict,
    ):
        return [
            label
            for label, _
            in sorted(
                category_index.items(),
                key=lambda x: x[1],
            )
        ]

    return list(category_index)


def fetch_eurostat_single_series(
    dataset_code,
    filters,
    value_name,
    force_refresh=False,
):
    """
    Fetch a Eurostat JSON-stat series where all dimensions
    other than time are reduced to one category.
    """

    EUROSTAT_CACHE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    filter_string = "_".join(
        f"{key}-{value}"
        for key, value
        in sorted(filters.items())
    )

    cache_path = (
        EUROSTAT_CACHE_DIR
        / f"{dataset_code}_{filter_string}.csv"
    )

    if (
        cache_path.exists()
        and not force_refresh
    ):
        return pd.read_csv(
            cache_path
        )

    url = (
        "https://ec.europa.eu/"
        "eurostat/api/dissemination/"
        "statistics/1.0/data/"
        f"{dataset_code}"
    )

    response = HTTP_SESSION.get(
        url,
        params={
            "lang": "EN",
            **filters,
        },
        timeout=(10, 180),
    )

    response.raise_for_status()

    data = response.json()

    dimension_ids = data["id"]
    dimension_sizes = data["size"]

    if "time" not in dimension_ids:
        raise ValueError(
            "Eurostat response has no time dimension."
        )

    time_position = (
        dimension_ids.index("time")
    )

    for i, dimension in enumerate(
        dimension_ids
    ):
        if (
            dimension != "time"
            and dimension_sizes[i] != 1
        ):
            raise ValueError(
                f"Dimension {dimension!r} "
                "was not reduced to one category."
            )

    years = _jsonstat_category_labels(
        data[
            "dimension"
        ][
            "time"
        ][
            "category"
        ][
            "index"
        ]
    )

    values = data["value"]

    rows = []

    for time_coordinate, year in enumerate(
        years
    ):
        coordinates = [
            0
            for _ in dimension_ids
        ]

        coordinates[
            time_position
        ] = time_coordinate

        flat_index = int(
            np.ravel_multi_index(
                tuple(coordinates),
                tuple(dimension_sizes),
                order="C",
            )
        )

        if isinstance(values, dict):
            value = values.get(
                str(flat_index),
                np.nan,
            )
        else:
            value = (
                values[flat_index]
                if flat_index < len(values)
                else np.nan
            )

        rows.append({
            "year": int(year),
            value_name: value,
        })

    result = pd.DataFrame(rows)

    result[value_name] = pd.to_numeric(
        result[value_name],
        errors="coerce",
    )

    result.to_csv(
        cache_path,
        index=False,
    )

    return result

In [ ]:
def fetch_ameco_primary_balance(
    ameco_country_code,
    force_refresh=False,
):
    """
    Fetch AMECO UBLGI:
    general-government primary balance as % GDP.
    """

    AMECO_CACHE_DIR.mkdir(
        parents=True,
        exist_ok=True,
    )

    cache_path = (
        AMECO_CACHE_DIR
        / (
            f"primary_balance_"
            f"{ameco_country_code}.csv"
        )
    )

    if (
        cache_path.exists()
        and not force_refresh
    ):
        return pd.read_csv(
            cache_path
        )

    series_code = (
        f"{ameco_country_code}."
        "1.0.319.0.UBLGI"
    )

    url = (
        "https://api.db.nomics.world/"
        "v22/series/"
        f"AMECO/UBLGI/{series_code}"
    )

    response = HTTP_SESSION.get(
        url,
        params={"observations": 1},
        timeout=(10, 180),
    )

    response.raise_for_status()

    payload = response.json()

    docs = (
        payload
        .get("series", {})
        .get("docs", [])
    )

    if not docs:
        raise ValueError(
            f"No AMECO primary-balance "
            f"series for {ameco_country_code}."
        )

    series = docs[0]

    result = pd.DataFrame({
        "year": pd.to_numeric(
            series["period"],
            errors="coerce",
        ),
        "primary_balance_gdp": pd.to_numeric(
            series["value"],
            errors="coerce",
        ),
    })

    result = (
        result
        .dropna()
        .copy()
    )

    result["year"] = (
        result["year"]
        .astype(int)
    )

    result["primary_balance_gdp"] /= 100

    result.to_csv(
        cache_path,
        index=False,
    )

    return result

## 4. Build the sovereign panel

All model variables are stored as decimals:

- `1.20` debt/GDP = 120%;
- `0.03` spread = 3%;
- `-0.02` primary balance = -2% of GDP.

In [ ]:
bund_monthly = fetch_ecb_10y(
    GERMANY_ECB_CODE,
)

bund_annual = (
    monthly_rate_to_annual(
        bund_monthly,
        "bund_10y",
    )
)

bund_annual = (
    bund_annual[
        bund_annual["year"].between(
            PANEL_START_YEAR,
            PANEL_END_YEAR,
        )
    ]
    .reset_index(drop=True)
)

bund_annual.head()

In [ ]:
def build_country_panel(
    country_name,
    codes,
):
    """
    Build one country's annual sovereign-risk panel.
    """

    rate_monthly = fetch_ecb_10y(
        codes["ecb"]
    )

    rate_annual = (
        monthly_rate_to_annual(
            rate_monthly,
            "sovereign_10y",
        )
    )

    rates = (
        rate_annual
        .merge(
            bund_annual,
            on="year",
            how="inner",
            validate="one_to_one",
        )
    )

    rates["spread_10y"] = (
        rates["sovereign_10y"]
        - rates["bund_10y"]
    )

    debt = (
        fetch_eurostat_single_series(
            dataset_code="gov_10dd_edpt1",
            filters={
                "freq": "A",
                "unit": "PC_GDP",
                "sector": "S13",
                "na_item": "GD",
                "geo": codes["eurostat"],
            },
            value_name="debt_gdp",
        )
    )

    debt["debt_gdp"] /= 100

    growth = (
        fetch_eurostat_single_series(
            dataset_code="nama_10_gdp",
            filters={
                "freq": "A",
                "unit": "CLV_PCH_PRE",
                "na_item": "B1GQ",
                "geo": codes["eurostat"],
            },
            value_name="real_growth",
        )
    )

    growth["real_growth"] /= 100

    primary_balance = (
        fetch_ameco_primary_balance(
            codes["ameco"]
        )
    )

    result = (
        rates
        .merge(
            debt,
            on="year",
            how="inner",
        )
        .merge(
            growth,
            on="year",
            how="inner",
        )
        .merge(
            primary_balance,
            on="year",
            how="inner",
        )
    )

    result = (
        result[
            result["year"].between(
                PANEL_START_YEAR,
                PANEL_END_YEAR,
            )
        ]
        .copy()
    )

    result["country"] = country_name

    return result[
        [
            "country",
            "year",
            "sovereign_10y",
            "bund_10y",
            "spread_10y",
            "debt_gdp",
            "primary_balance_gdp",
            "real_growth",
        ]
    ]


panel_parts = []

for country_name, codes in (
    COUNTRIES.items()
):
    print(
        f"Building {country_name}..."
    )

    panel_parts.append(
        build_country_panel(
            country_name,
            codes,
        )
    )

sovereign_panel = (
    pd.concat(
        panel_parts,
        ignore_index=True,
    )
    .sort_values(
        [
            "country",
            "year",
        ]
    )
    .reset_index(drop=True)
)

sovereign_panel

In [ ]:
required_columns = [
    "country",
    "year",
    "spread_10y",
    "debt_gdp",
    "primary_balance_gdp",
    "real_growth",
]

assert not (
    sovereign_panel[
        required_columns
    ]
    .isna()
    .any()
    .any()
), (
    "NULL values remain in the merged panel."
)

assert not sovereign_panel.duplicated(
    subset=[
        "country",
        "year",
    ]
).any(), (
    "Duplicate country-year observations found."
)

panel_counts = (
    sovereign_panel
    .groupby("country")
    .size()
    .sort_values(
        ascending=False
    )
    .rename("observations")
)

print(panel_counts)

assert (
    panel_counts >= 15
).all(), (
    "At least one country has fewer than "
    "15 usable annual observations."
)

panel_path = (
    STRESS_DIR
    / "euro_area_sovereign_panel.csv"
)

sovereign_panel.to_csv(
    panel_path,
    index=False,
)

print(
    "\nPanel saved to:",
    panel_path,
)

## 5. Two-way fixed-effects spread regression

Debt enters with a one-year lag.

The model includes country fixed effects and year fixed effects, with standard errors clustered by country.

In [ ]:
regression_panel = (
    sovereign_panel
    .copy()
)

regression_panel["debt_gdp_lag"] = (
    regression_panel
    .groupby("country")[
        "debt_gdp"
    ]
    .shift(1)
)

regression_panel = (
    regression_panel
    .dropna(
        subset=[
            "spread_10y",
            "debt_gdp_lag",
            "primary_balance_gdp",
            "real_growth",
        ]
    )
    .reset_index(drop=True)
)

print(
    "Regression observations:",
    len(regression_panel),
)

print(
    "Countries:",
    regression_panel["country"].nunique(),
)

print(
    "Years:",
    regression_panel["year"].min(),
    "to",
    regression_panel["year"].max(),
)

In [ ]:
SPREAD_FORMULA = (
    "spread_10y ~ "
    "debt_gdp_lag "
    "+ primary_balance_gdp "
    "+ real_growth "
    "+ C(country) "
    "+ C(year)"
)

spread_model = (
    smf.ols(
        formula=SPREAD_FORMULA,
        data=regression_panel,
    )
    .fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                regression_panel["country"]
        },
    )
)

print(
    spread_model.summary()
)

## 6. Translate the coefficients into economically readable effects

Because all variables are decimals:

- +10 percentage points of debt/GDP = `+0.10`;
- +1 percentage point of primary balance = `+0.01`;
- -1 percentage point of real growth = `-0.01`.

Spread effects are reported in basis points.

In [ ]:
beta_debt = (
    spread_model.params[
        "debt_gdp_lag"
    ]
)

beta_pb = (
    spread_model.params[
        "primary_balance_gdp"
    ]
)

beta_growth = (
    spread_model.params[
        "real_growth"
    ]
)

economic_effects = pd.DataFrame({
    "shock": [
        "+10 pp debt/GDP",
        "+1 pp primary balance/GDP",
        "-1 pp real GDP growth",
    ],
    "spread_effect_bp": [
        beta_debt
        * 0.10
        * 10_000,

        beta_pb
        * 0.01
        * 10_000,

        beta_growth
        * (-0.01)
        * 10_000,
    ],
})

economic_effects.round(2)

In [ ]:
panel_no_greece = (
    regression_panel[
        regression_panel["country"]
        != "Greece"
    ]
    .copy()
)

spread_model_no_greece = (
    smf.ols(
        formula=SPREAD_FORMULA,
        data=panel_no_greece,
    )
    .fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                panel_no_greece["country"]
        },
    )
)

coefficient_comparison = pd.DataFrame({
    "coefficient": [
        "debt_gdp_lag",
        "primary_balance_gdp",
        "real_growth",
    ],
    "full_panel": [
        spread_model.params[
            "debt_gdp_lag"
        ],
        spread_model.params[
            "primary_balance_gdp"
        ],
        spread_model.params[
            "real_growth"
        ],
    ],
    "exclude_greece": [
        spread_model_no_greece.params[
            "debt_gdp_lag"
        ],
        spread_model_no_greece.params[
            "primary_balance_gdp"
        ],
        spread_model_no_greece.params[
            "real_growth"
        ],
    ],
})

coefficient_comparison[
    "full_panel_effect_bp"
] = [
    coefficient_comparison.loc[
        0,
        "full_panel",
    ]
    * 0.10
    * 10_000,

    coefficient_comparison.loc[
        1,
        "full_panel",
    ]
    * 0.01
    * 10_000,

    coefficient_comparison.loc[
        2,
        "full_panel",
    ]
    * (-0.01)
    * 10_000,
]

coefficient_comparison[
    "exclude_greece_effect_bp"
] = [
    coefficient_comparison.loc[
        0,
        "exclude_greece",
    ]
    * 0.10
    * 10_000,

    coefficient_comparison.loc[
        1,
        "exclude_greece",
    ]
    * 0.01
    * 10_000,

    coefficient_comparison.loc[
        2,
        "exclude_greece",
    ]
    * (-0.01)
    * 10_000,
]

coefficient_comparison.round(3)

In [ ]:
regression_panel[
    "fitted_spread_10y"
] = (
    spread_model.predict(
        regression_panel
    )
)

regression_panel[
    "residual_spread_10y"
] = (
    regression_panel[
        "spread_10y"
    ]
    -
    regression_panel[
        "fitted_spread_10y"
    ]
)

fig, ax = plt.subplots(
    figsize=(8, 6)
)

ax.scatter(
    regression_panel[
        "fitted_spread_10y"
    ]
    * 10_000,
    regression_panel[
        "spread_10y"
    ]
    * 10_000,
    alpha=0.6,
)

lims = [
    min(
        regression_panel[
            "fitted_spread_10y"
        ].min(),
        regression_panel[
            "spread_10y"
        ].min(),
    )
    * 10_000,

    max(
        regression_panel[
            "fitted_spread_10y"
        ].max(),
        regression_panel[
            "spread_10y"
        ].max(),
    )
    * 10_000,
]

ax.plot(
    lims,
    lims,
    linestyle="--",
)

ax.set_title(
    "Observed vs fitted euro-area sovereign spreads"
)

ax.set_xlabel(
    "Fitted spread (bp)"
)

ax.set_ylabel(
    "Observed spread (bp)"
)

ax.grid(alpha=0.25)

plt.show()

# Current stopping point

Before implementing the recursive sovereign-stress simulator, inspect:

1. the sign and magnitude of `beta_debt`;
2. the sign and magnitude of `beta_pb`;
3. the sign and magnitude of `beta_growth`;
4. whether those coefficients remain broadly similar when Greece is excluded;
5. the observed-vs-fitted spread diagnostics.

We should **not** hard-code the debt-to-spread feedback until these estimates have been inspected.

The next stage will convert accepted coefficients into a France-specific deviation equation:

\[
\Delta s_t
=
\beta_d
(d_{t-1}-d_{t-1}^{central})
+
\beta_{pb}
(pb_t-pb_t^{central})
+
\beta_g
(g_t-g_t^{central})
\]

and feed that spread adjustment recursively into the AFT refinancing engine and general-government DSA.

# 7. France 10Y → 30Y sovereign-spread transmission

The euro-area panel estimates how fiscal and macro fundamentals affect the **10Y sovereign spread**. The AFT refinancing engine, however, is anchored on the **30Y OAT**.

We therefore estimate a France-specific maturity-transmission equation:

\[
\Delta s^{30Y}_t
=
\alpha
+
\lambda
\Delta s^{10Y}_t
+
\eta_t
\]

where:

\[
s^{10Y}_t
=
TEC10^{FR}_t
-
Bund10^{DE}_t
\]

and:

\[
s^{30Y}_t
=
TEC30^{FR}_t
-
Bund30^{DE}_t.
\]

The regression is estimated in **monthly changes**, rather than spread levels, to avoid a spuriously strong relationship caused by persistence.

For the recursive DSA, only the slope \(\lambda\) will be used:

\[
\Delta s^{30Y,\ feedback}_t
=
\lambda
\Delta s^{10Y,\ feedback}_t.
\]

The estimated intercept is not applied to the feedback term: zero endogenous 10Y feedback must imply zero endogenous 30Y feedback.

In [ ]:
from io import BytesIO


# ============================================================
# Official source series
# ============================================================

# Banque de France combined CNO-TEC historical selection.
BDF_TEC_SELECTION_PAGE = (
    "https://webstat.banque-france.fr/"
    "fr/selection/5385693/"
)

BDF_TEC_EXPORT_URLS = [
    (
        "https://webstat.banque-france.fr/"
        "export/csv-columns/fr/selection/5385693"
    ),
    (
        "https://webstat.banque-france.fr/"
        "fr/downloadFile.do"
        "?id=5385693&exportType=csv"
    ),
]

# Deutsche Bundesbank:
# yields derived from the Svensson term structure on listed
# Federal securities, exact residual maturities.
BUND_10Y_KEY = (
    "D.I.ZAR.ZI.EUR.S1311.B.A604."
    "R10XX.R.A.A._Z._Z.A"
)

BUND_30Y_KEY = (
    "D.I.ZAR.ZI.EUR.S1311.B.A604."
    "R30XX.R.A.A._Z._Z.A"
)

BUNDESBANK_API_BASE = (
    "https://api.statistiken.bundesbank.de/"
    "rest/data/BBSIS"
)

In [ ]:
# ============================================================
# Generic CSV helpers
# ============================================================

import unicodedata


def read_csv_bytes_auto(content):
    """Read a CSV payload using common European encodings."""
    last_error = None

    for encoding in [
        "utf-8-sig",
        "utf-8",
        "latin1",
    ]:
        try:
            return pd.read_csv(
                BytesIO(content),
                sep=None,
                engine="python",
                encoding=encoding,
            )
        except Exception as exc:
            last_error = exc

    raise ValueError(
        "Could not parse CSV payload."
    ) from last_error


def normalize_column_name(value):
    """
    Normalize labels robustly, including French accents.
    """
    text = str(
        value
    ).strip()

    text = (
        unicodedata
        .normalize(
            "NFKD",
            text,
        )
        .encode(
            "ascii",
            "ignore",
        )
        .decode(
            "ascii",
        )
    )

    text = (
        text
        .lower()
        .replace(
            "€",
            "e",
        )
        .replace(
            " ",
            "_",
        )
        .replace(
            "-",
            "_",
        )
    )

    while "__" in text:
        text = text.replace(
            "__",
            "_",
        )

    return text


def find_date_column(df):
    normalized = {
        column: normalize_column_name(column)
        for column in df.columns
    }

    for pattern in [
        "time_period",
        "date",
        "periode",
        "period",
    ]:
        for column, normalized_name in normalized.items():
            if pattern in normalized_name:
                return column

    best_column = None
    best_share = 0.0

    for column in df.columns:
        parsed = pd.to_datetime(
            df[column],
            errors="coerce",
            dayfirst=True,
        )

        share = parsed.notna().mean()

        if share > best_share:
            best_share = share
            best_column = column

    if (
        best_column is None
        or best_share < 0.50
    ):
        raise ValueError(
            "Could not identify a date column."
        )

    return best_column


def numeric_series(series):
    """Convert decimal-point or decimal-comma text to numeric."""
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(
            series,
            errors="coerce",
        )

    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.replace(
            "\u00a0",
            "",
            regex=False,
        )
        .str.replace(
            ",",
            ".",
            regex=False,
        )
    )

    return pd.to_numeric(
        cleaned,
        errors="coerce",
    )

In [ ]:
# ============================================================
# Banque de France TEC10 / TEC30 history
# ============================================================

def fetch_bdf_tec_history(
    force_refresh=False,
):
    """
    Download the combined Banque de France CNO-TEC history
    and return daily TEC10 and TEC30 yields.

    Source yields are percent per annum and are converted
    to decimals.
    """

    cache_dir = (
        STRESS_RAW_DIR
        / "banque_de_france"
    )

    cache_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    cache_path = (
        cache_dir
        / "tec10_tec30_daily.csv"
    )

    if (
        cache_path.exists()
        and not force_refresh
    ):
        return pd.read_csv(
            cache_path,
            parse_dates=["date"],
        )

    session = create_retry_session()

    try:
        session.get(
            BDF_TEC_SELECTION_PAGE,
            timeout=(10, 60),
        )
    except requests.RequestException:
        pass

    raw_df = None
    errors = []

    for url in BDF_TEC_EXPORT_URLS:
        try:
            response = session.get(
                url,
                headers={
                    "Referer":
                        BDF_TEC_SELECTION_PAGE,
                },
                timeout=(10, 180),
            )

            response.raise_for_status()

            first_bytes = (
                response.content[:200]
                .lower()
            )

            if (
                b"<html" in first_bytes
                or b"<!doctype" in first_bytes
            ):
                raise ValueError(
                    "Banque de France returned HTML "
                    "instead of a CSV."
                )

            candidate = (
                read_csv_bytes_auto(
                    response.content
                )
            )

            if len(candidate) > 0:
                raw_df = candidate
                break

        except Exception as exc:
            errors.append(
                (
                    url,
                    repr(exc),
                )
            )

    if raw_df is None:
        raise RuntimeError(
            "Could not download Banque de France TEC history.\n"
            + "\n".join(
                f"- {url}: {error}"
                for url, error in errors
            )
        )

    print(
        "Banque de France raw columns:"
    )
    print(
        raw_df.columns.tolist()
    )

    date_column = (
        find_date_column(
            raw_df
        )
    )

    normalized = {
        column:
            normalize_column_name(
                column
            )
        for column in raw_df.columns
    }


def find_tec_column(
    maturity,
):
    """
    Identify Banque de France constant-maturity columns.

    Handles both descriptive labels such as
    'Taux de l'Echéance Constante - 10 ans'
    and shorter TEC10 / TEC30-style labels.
    """
    maturity = int(
        maturity
    )

    candidates = []

    for column, name in (
        normalized.items()
    ):
        is_constant_maturity = (
            "echeance_constante"
            in name
        )

        maturity_patterns = [
            f"_{maturity}_an",
            f"_{maturity}_ans",
        ]

        has_maturity = any(
            pattern in name
            for pattern
            in maturity_patterns
        )

        compact = (
            name
            .replace(
                "_",
                "",
            )
            .replace(
                "'",
                "",
            )
        )

        has_tec_label = any(
            pattern in compact
            for pattern
            in [
                f"tec{maturity}",
                f"moytec{maturity}",
                f"frmoytec{maturity}",
            ]
        )

        if (
            (
                is_constant_maturity
                and has_maturity
            )
            or has_tec_label
        ):
            candidates.append(
                column
            )

    if not candidates:
        raise ValueError(
            f"Could not identify TEC{maturity}. "
            f"Columns: {raw_df.columns.tolist()}"
        )

    return candidates[0]

    tec10_column = (
        find_tec_column(10)
    )

    tec30_column = (
        find_tec_column(30)
    )

    result = pd.DataFrame({
        "date":
            pd.to_datetime(
                raw_df[date_column],
                errors="coerce",
                dayfirst=True,
            ),

        "france_10y":
            numeric_series(
                raw_df[tec10_column]
            )
            / 100,

        "france_30y":
            numeric_series(
                raw_df[tec30_column]
            )
            / 100,
    })

    result = (
        result
        .dropna()
        .drop_duplicates(
            subset=["date"]
        )
        .sort_values("date")
        .reset_index(drop=True)
    )

    if len(result) < 500:
        raise ValueError(
            "Unexpectedly few Banque de France "
            f"TEC observations: {len(result)}"
        )

    result.to_csv(
        cache_path,
        index=False,
    )

    return result

In [ ]:
# ============================================================
# Deutsche Bundesbank exact-maturity yields
# ============================================================

def fetch_bundesbank_term_yield(
    series_key,
    value_name,
    force_refresh=False,
):
    """
    Fetch a Bundesbank BBSIS daily yield series.

    Source units are percent per annum and are converted
    to decimals.
    """

    cache_dir = (
        STRESS_RAW_DIR
        / "bundesbank"
    )

    cache_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    safe_key = (
        series_key
        .replace(".", "_")
    )

    cache_path = (
        cache_dir
        / f"{safe_key}.csv"
    )

    if (
        cache_path.exists()
        and not force_refresh
    ):
        cached = pd.read_csv(
            cache_path,
            parse_dates=["date"],
        )

        return cached[
            [
                "date",
                value_name,
            ]
        ]

    url = (
        f"{BUNDESBANK_API_BASE}/"
        f"{series_key}"
    )

    response = HTTP_SESSION.get(
        url,
        params={
            "format": "csv",
            "lang": "en",
        },
        timeout=(10, 180),
    )

    response.raise_for_status()

    raw = (
        read_csv_bytes_auto(
            response.content
        )
    )

    print(
        f"Bundesbank {value_name} raw columns:"
    )
    print(
        raw.columns.tolist()
    )

    date_column = (
        find_date_column(
            raw
        )
    )

    normalized = {
        column:
            normalize_column_name(
                column
            )
        for column in raw.columns
    }

    value_candidates = [
        column
        for column, name
        in normalized.items()
        if (
            "obs_value" in name
            or name == "value"
            or "value" in name
        )
    ]

    if not value_candidates:
        candidate_scores = []

        for column in raw.columns:
            if column == date_column:
                continue

            numeric = (
                numeric_series(
                    raw[column]
                )
            )

            candidate_scores.append(
                (
                    numeric.notna().sum(),
                    column,
                )
            )

        candidate_scores.sort(
            reverse=True
        )

        if not candidate_scores:
            raise ValueError(
                "Could not identify Bundesbank value column."
            )

        value_column = (
            candidate_scores[0][1]
        )

    else:
        value_column = (
            value_candidates[0]
        )

    result = pd.DataFrame({
        "date":
            pd.to_datetime(
                raw[date_column],
                errors="coerce",
                dayfirst=True,
            ),

        value_name:
            numeric_series(
                raw[value_column]
            )
            / 100,
    })

    result = (
        result
        .dropna()
        .drop_duplicates(
            subset=["date"]
        )
        .sort_values("date")
        .reset_index(drop=True)
    )

    result.to_csv(
        cache_path,
        index=False,
    )

    return result

In [ ]:
# ============================================================
# Download the four maturity series
# ============================================================

france_term_yields = (
    fetch_bdf_tec_history()
)

bund_10y_daily = (
    fetch_bundesbank_term_yield(
        BUND_10Y_KEY,
        "germany_10y",
    )
)

bund_30y_daily = (
    fetch_bundesbank_term_yield(
        BUND_30Y_KEY,
        "germany_30y",
    )
)

print(
    "France history:",
    france_term_yields["date"].min(),
    "to",
    france_term_yields["date"].max(),
)

print(
    "Germany 10Y:",
    bund_10y_daily["date"].min(),
    "to",
    bund_10y_daily["date"].max(),
)

print(
    "Germany 30Y:",
    bund_30y_daily["date"].min(),
    "to",
    bund_30y_daily["date"].max(),
)

## 8. Construct matched monthly 10Y and 30Y France–Germany spreads

We use monthly averages of the daily constant/exact-maturity yields. This avoids requiring identical French and German trading calendars and reduces day-to-day microstructure noise.

The transmission regression is estimated on first differences:

\[
\Delta s^{30Y}_t
=
\alpha
+
\lambda
\Delta s^{10Y}_t
+
\eta_t.
\]

In [ ]:
def daily_to_monthly_average(
    df,
    value_columns,
):
    result = (
        df
        .copy()
        .set_index("date")[
            list(value_columns)
        ]
        .resample("ME")
        .mean()
        .reset_index()
    )

    result["month"] = (
        result["date"]
        .dt.to_period("M")
    )

    return result.drop(
        columns=["date"]
    )


france_monthly = (
    daily_to_monthly_average(
        france_term_yields,
        [
            "france_10y",
            "france_30y",
        ],
    )
)

germany_10y_monthly = (
    daily_to_monthly_average(
        bund_10y_daily,
        ["germany_10y"],
    )
)

germany_30y_monthly = (
    daily_to_monthly_average(
        bund_30y_daily,
        ["germany_30y"],
    )
)

maturity_panel = (
    france_monthly
    .merge(
        germany_10y_monthly,
        on="month",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        germany_30y_monthly,
        on="month",
        how="inner",
        validate="one_to_one",
    )
    .sort_values("month")
    .reset_index(drop=True)
)

maturity_panel["spread_10y"] = (
    maturity_panel["france_10y"]
    - maturity_panel["germany_10y"]
)

maturity_panel["spread_30y"] = (
    maturity_panel["france_30y"]
    - maturity_panel["germany_30y"]
)

maturity_panel["delta_spread_10y"] = (
    maturity_panel["spread_10y"]
    .diff()
)

maturity_panel["delta_spread_30y"] = (
    maturity_panel["spread_30y"]
    .diff()
)

transmission_sample = (
    maturity_panel
    .dropna(
        subset=[
            "delta_spread_10y",
            "delta_spread_30y",
        ]
    )
    .copy()
)

print(
    "Transmission sample:",
    transmission_sample["month"].min(),
    "to",
    transmission_sample["month"].max(),
)

print(
    "Monthly observations:",
    len(transmission_sample),
)

transmission_sample[
    [
        "month",
        "spread_10y",
        "spread_30y",
        "delta_spread_10y",
        "delta_spread_30y",
    ]
].tail()

In [ ]:
# ============================================================
# 10Y -> 30Y transmission regression
# ============================================================

MATURITY_TRANSMISSION_FORMULA = (
    "delta_spread_30y ~ "
    "delta_spread_10y"
)

maturity_transmission_model = (
    smf.ols(
        formula=MATURITY_TRANSMISSION_FORMULA,
        data=transmission_sample,
    )
    .fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": 3,
        },
    )
)

print(
    maturity_transmission_model.summary()
)

In [ ]:
# ============================================================
# Economically readable transmission coefficient
# ============================================================

LAMBDA_10Y_TO_30Y = float(
    maturity_transmission_model.params[
        "delta_spread_10y"
    ]
)

lambda_se = float(
    maturity_transmission_model.bse[
        "delta_spread_10y"
    ]
)

lambda_ci = (
    maturity_transmission_model
    .conf_int()
    .loc[
        "delta_spread_10y"
    ]
)

transmission_summary = pd.DataFrame({
    "metric": [
        "lambda_10y_to_30y",
        "standard_error",
        "95pct_ci_low",
        "95pct_ci_high",
        "r_squared",
        "n_observations",
        "30Y effect of +100bp 10Y shock",
    ],

    "value": [
        LAMBDA_10Y_TO_30Y,
        lambda_se,
        float(lambda_ci.iloc[0]),
        float(lambda_ci.iloc[1]),
        float(
            maturity_transmission_model.rsquared
        ),
        int(
            maturity_transmission_model.nobs
        ),
        (
            LAMBDA_10Y_TO_30Y
            * 100
        ),
    ],
})

transmission_summary

In [ ]:
# ============================================================
# Transmission scatter
# ============================================================

x_bp = (
    transmission_sample[
        "delta_spread_10y"
    ]
    .to_numpy()
    * 10_000
)

y_bp = (
    transmission_sample[
        "delta_spread_30y"
    ]
    .to_numpy()
    * 10_000
)

x_grid = np.linspace(
    x_bp.min(),
    x_bp.max(),
    200,
)

intercept_bp = (
    maturity_transmission_model.params[
        "Intercept"
    ]
    * 10_000
)

fitted_grid = (
    intercept_bp
    +
    LAMBDA_10Y_TO_30Y
    * x_grid
)

fig, ax = plt.subplots(
    figsize=(9, 6)
)

ax.scatter(
    x_bp,
    y_bp,
    alpha=0.45,
    label="Monthly observations",
)

ax.plot(
    x_grid,
    fitted_grid,
    linewidth=2,
    label=(
        "HAC OLS fit "
        f"(λ={LAMBDA_10Y_TO_30Y:.2f})"
    ),
)

ax.axhline(
    0,
    linewidth=0.8,
)

ax.axvline(
    0,
    linewidth=0.8,
)

ax.set_title(
    "France sovereign-spread transmission: 10Y → 30Y"
)

ax.set_xlabel(
    "Monthly change in France–Germany 10Y spread (bp)"
)

ax.set_ylabel(
    "Monthly change in France–Germany 30Y spread (bp)"
)

ax.grid(alpha=0.25)
ax.legend()

plt.show()

In [ ]:
# ============================================================
# Stability diagnostics
# ============================================================

def estimate_lambda_for_sample(
    sample,
    label,
):
    model = (
        smf.ols(
            formula=MATURITY_TRANSMISSION_FORMULA,
            data=sample,
        )
        .fit(
            cov_type="HAC",
            cov_kwds={
                "maxlags": 3,
            },
        )
    )

    ci = (
        model
        .conf_int()
        .loc[
            "delta_spread_10y"
        ]
    )

    return {
        "sample": label,
        "n": int(model.nobs),
        "lambda": float(
            model.params[
                "delta_spread_10y"
            ]
        ),
        "se": float(
            model.bse[
                "delta_spread_10y"
            ]
        ),
        "ci_low": float(
            ci.iloc[0]
        ),
        "ci_high": float(
            ci.iloc[1]
        ),
        "r_squared": float(
            model.rsquared
        ),
    }


sample_midpoint = (
    transmission_sample["month"]
    .sort_values()
    .iloc[
        len(transmission_sample)
        // 2
    ]
)

transmission_stability = pd.DataFrame([
    estimate_lambda_for_sample(
        transmission_sample,
        "Full sample",
    ),

    estimate_lambda_for_sample(
        transmission_sample[
            transmission_sample["month"]
            <= sample_midpoint
        ],
        "First half",
    ),

    estimate_lambda_for_sample(
        transmission_sample[
            transmission_sample["month"]
            > sample_midpoint
        ],
        "Second half",
    ),
])

transmission_stability.round(3)

In [ ]:
# ============================================================
# Save maturity-transmission calibration
# ============================================================

transmission_sample.assign(
    month=lambda df:
        df["month"].astype(str)
).to_csv(
    STRESS_DIR
    / "france_10y_30y_spread_history.csv",
    index=False,
)

transmission_summary.to_csv(
    STRESS_DIR
    / "maturity_transmission_summary.csv",
    index=False,
)

transmission_stability.to_csv(
    STRESS_DIR
    / "maturity_transmission_stability.csv",
    index=False,
)

print(
    "Saved maturity-transmission outputs to:",
    STRESS_DIR,
)

# Next decision

Before wiring the feedback loop into the AFT portfolio, inspect:

1. `LAMBDA_10Y_TO_30Y`;
2. its HAC 95% confidence interval;
3. the regression \(R^2\);
4. whether the first-half and second-half estimates are reasonably stable;
5. the scatter of monthly 10Y and 30Y spread changes.

If the transmission estimate is credible, the recursive sovereign-stress model will use:

\[
\Delta s^{10Y,\ feedback}_t
=
\beta_1
(d_{t-1}-d^C_{t-1})
+
\beta_2
(d_{t-1}^2-(d^C_{t-1})^2)
+
\beta_{pb}
(pb_t-pb^C_t)
+
\beta_g
(g_t-g^C_t)
\]

followed by:

\[
\Delta s^{30Y,\ feedback}_t
=
\lambda
\Delta s^{10Y,\ feedback}_t.
\]

That 30Y feedback shock will then alter the OAT anchor, the full issuance curve, AFT refinancing costs, general-government interest expenditure, and finally next year's debt ratio.

# 9. Recursive sovereign-stress engine

We now connect the empirically calibrated spread mechanism to the AFT refinancing engine.

For a given ordinary-times Notebook 02 path, define the **stress deviation** relative to that same path:

\[
\Delta s^{10Y}_t
=
\beta_1(d^{S}_{t-1}-d^{B}_{t-1})
+
\beta_2[(d^{S}_{t-1})^2-(d^{B}_{t-1})^2]
+
\beta_{pb}(pb^{S}_t-pb^{B}_t)
+
\beta_g(g^{S}_t-g^{B}_t).
\]

Then:

\[
\Delta s^{30Y}_t
=
\lambda \Delta s^{10Y}_t
\]

and:

\[
OAT^{30Y,S}_t
=
OAT^{30Y,B}_t
+
\Delta s^{30Y}_t.
\]

Using the ordinary-times path itself as the reference has an important property:

> If there is no initial debt deviation and no fiscal/growth deviation, the sovereign-feedback extension must reproduce the Notebook 02 path exactly.

The first validation below therefore runs a **zero-feedback identity test**, followed by a +10 percentage-point initial-debt impulse response. The latter is a diagnostic, not a forecast.

In [ ]:
# ============================================================
# Recreate the calibrated quadratic spread model
# ============================================================

regression_panel[
    "debt_gdp_lag_sq"
] = (
    regression_panel[
        "debt_gdp_lag"
    ]
    ** 2
)

SPREAD_FORMULA_QUADRATIC = (
    "spread_10y ~ "
    "debt_gdp_lag "
    "+ debt_gdp_lag_sq "
    "+ primary_balance_gdp "
    "+ real_growth "
    "+ C(country) "
    "+ C(year)"
)

spread_model_quadratic = (
    smf.ols(
        formula=SPREAD_FORMULA_QUADRATIC,
        data=regression_panel,
    )
    .fit(
        cov_type="cluster",
        cov_kwds={
            "groups":
                regression_panel[
                    "country"
                ]
        },
    )
)

BETA_DEBT_LINEAR = float(
    spread_model_quadratic.params[
        "debt_gdp_lag"
    ]
)

BETA_DEBT_SQUARED = float(
    spread_model_quadratic.params[
        "debt_gdp_lag_sq"
    ]
)

BETA_PRIMARY_BALANCE = float(
    spread_model_quadratic.params[
        "primary_balance_gdp"
    ]
)

BETA_REAL_GROWTH = float(
    spread_model_quadratic.params[
        "real_growth"
    ]
)

LAMBDA_10Y_TO_30Y = float(
    maturity_transmission_model.params[
        "delta_spread_10y"
    ]
)

feedback_parameters = pd.Series({
    "beta_debt_linear":
        BETA_DEBT_LINEAR,
    "beta_debt_squared":
        BETA_DEBT_SQUARED,
    "beta_primary_balance":
        BETA_PRIMARY_BALANCE,
    "beta_real_growth":
        BETA_REAL_GROWTH,
    "lambda_10y_to_30y":
        LAMBDA_10Y_TO_30Y,
})

feedback_parameters

In [ ]:
# ============================================================
# AFT setup reused from Notebook 02
# ============================================================

from src.models.debt_portfolio import (
    load_portfolio,
    simulate_portfolio_path,
    curve_from_30y_anchor,
)

SNAPSHOT_DATE = pd.Timestamp(
    "2026-10-03"
)

CURRENT_YEAR = 2026
CURRENT_OAT_30Y = 0.0517

# The yield-curve bridge is anchored to the observed
# October-2026 30Y OAT level.
BASELINE_OAT_30Y = (
    CURRENT_OAT_30Y
)

PORTFOLIO_PATH = (
    PROCESSED_DIR
    / f"aft_portfolio_{SNAPSHOT_DATE.date()}.csv"
)

portfolio = load_portfolio(
    PORTFOLIO_PATH,
    snapshot_date=SNAPSHOT_DATE,
)

INITIAL_GDP_EUR = 3_056e9
INITIAL_DEBT_GDP = 1.185

INITIAL_GG_DEBT_EUR = (
    INITIAL_GDP_EUR
    * INITIAL_DEBT_GDP
)

YEARS = (
    mc02[
        "years"
    ]
    .astype(int)
    .tolist()
)

NOMINAL_CURVE_OFFSETS_TO_30Y = {
    1.0:  -0.0150,
    2.0:  -0.0125,
    5.0:  -0.0100,
    10.0: -0.0060,
    15.0: -0.0035,
    30.0:  0.0000,
    50.0:  0.0015,
}

# NOTE:
# These real linker curves remain illustrative calibration
# inputs. The nominal 30Y anchor is now aligned to the
# observed 2026 OAT level, but the real curves can later be
# replaced with observed OATi/OAT€i real-yield curves.
BASELINE_REAL_EURO_CURVE = {
    5.0: 0.020,
    10.0: 0.025,
    20.0: 0.028,
    30.0: 0.030,
}

BASELINE_REAL_FR_CURVE = {
    5.0: 0.021,
    10.0: 0.026,
    20.0: 0.029,
    30.0: 0.031,
}

ISSUANCE_PLAN = [
    {
        "security_type": "BTF",
        "maturity_years": 1.0,
        "weight": 0.10,
    },
    {
        "security_type": "OAT",
        "maturity_years": 2.0,
        "weight": 0.10,
    },
    {
        "security_type": "OAT",
        "maturity_years": 5.0,
        "weight": 0.18,
    },
    {
        "security_type": "OAT",
        "maturity_years": 10.0,
        "weight": 0.27,
    },
    {
        "security_type": "OATEI",
        "maturity_years": 10.0,
        "weight": 0.05,
    },
    {
        "security_type": "OATI",
        "maturity_years": 10.0,
        "weight": 0.03,
    },
    {
        "security_type": "OAT",
        "maturity_years": 15.0,
        "weight": 0.10,
    },
    {
        "security_type": "OAT",
        "maturity_years": 30.0,
        "weight": 0.12,
    },
    {
        "security_type": "OAT",
        "maturity_years": 50.0,
        "weight": 0.05,
    },
]

assert np.isclose(
    sum(
        item[
            "weight"
        ]
        for item
        in ISSUANCE_PLAN
    ),
    1.0,
)

required_mc02_arrays = {
    "real_growth",
    "inflation",
    "nominal_growth",
    "primary_balance_gdp",
    "oat_30y",
    "debt_gdp",
    "gg_effective_rate",
}

missing_mc02_arrays = (
    required_mc02_arrays
    - set(
        mc02.files
    )
)

if missing_mc02_arrays:
    raise ValueError(
        "Notebook 02 NPZ is missing the newly required arrays: "
        f"{sorted(missing_mc02_arrays)}. "
        "Rerun Notebook 02 through its save-output cell."
    )

In [ ]:
# ============================================================
# Feedback functions
# ============================================================

def calculate_10y_sovereign_feedback(
    stress_debt_gdp_previous,
    base_debt_gdp_previous,
    primary_balance_adjustment=0.0,
    real_growth_adjustment=0.0,
    feedback_scale=1.0,
):
    # Additional France-Germany 10Y spread generated by
    # deviations from the ordinary-times reference path.
    debt_effect = (
        BETA_DEBT_LINEAR
        * (
            stress_debt_gdp_previous
            - base_debt_gdp_previous
        )
        +
        BETA_DEBT_SQUARED
        * (
            stress_debt_gdp_previous ** 2
            - base_debt_gdp_previous ** 2
        )
    )

    fiscal_effect = (
        BETA_PRIMARY_BALANCE
        * primary_balance_adjustment
    )

    growth_effect = (
        BETA_REAL_GROWTH
        * real_growth_adjustment
    )

    return float(
        feedback_scale
        * (
            debt_effect
            + fiscal_effect
            + growth_effect
        )
    )


def transmit_feedback_to_30y(
    feedback_10y,
    transmission_lambda=LAMBDA_10Y_TO_30Y,
):
    return float(
        transmission_lambda
        * feedback_10y
    )


def market_curves_from_30y(
    oat_30y,
):
    nominal_curve = (
        curve_from_30y_anchor(
            oat_30y_yield=float(
                oat_30y
            ),
            offsets_to_30y=(
                NOMINAL_CURVE_OFFSETS_TO_30Y
            ),
        )
    )

    yield_shift = (
        float(
            oat_30y
        )
        - BASELINE_OAT_30Y
    )

    real_euro_curve = {
        maturity:
            rate
            + yield_shift
        for maturity, rate
        in BASELINE_REAL_EURO_CURVE.items()
    }

    real_fr_curve = {
        maturity:
            rate
            + yield_shift
        for maturity, rate
        in BASELINE_REAL_FR_CURVE.items()
    }

    return {
        "nominal":
            nominal_curve,
        "real_euro":
            real_euro_curve,
        "real_fr":
            real_fr_curve,
    }


assert np.isclose(
    calculate_10y_sovereign_feedback(
        stress_debt_gdp_previous=1.20,
        base_debt_gdp_previous=1.20,
    ),
    0.0,
)

print(
    "Sovereign-feedback helper validation passed."
)

# ============================================================
# Validate updated 30Y anchor
# ============================================================

assert np.isclose(
    BASELINE_OAT_30Y,
    CURRENT_OAT_30Y,
)

test_curves_at_anchor = (
    market_curves_from_30y(
        CURRENT_OAT_30Y
    )
)

assert np.isclose(
    test_curves_at_anchor[
        "nominal"
    ][
        30.0
    ],
    CURRENT_OAT_30Y,
)

assert np.isclose(
    test_curves_at_anchor[
        "real_euro"
    ][
        30.0
    ],
    BASELINE_REAL_EURO_CURVE[
        30.0
    ],
)

assert np.isclose(
    test_curves_at_anchor[
        "real_fr"
    ][
        30.0
    ],
    BASELINE_REAL_FR_CURVE[
        30.0
    ],
)

print(
    "Updated 30Y anchor validation passed:",
    f"{CURRENT_OAT_30Y:.2%}",
)

In [ ]:
# ============================================================
# Build one Notebook 02 ordinary-times reference path
# ============================================================

def build_notebook02_reference_path(
    path_index,
):
    n_paths = (
        mc02[
            "debt_gdp"
        ]
        .shape[0]
    )

    if not (
        0
        <= path_index
        < n_paths
    ):
        raise IndexError(
            f"path_index must be between 0 and {n_paths - 1}."
        )

    return pd.DataFrame({
        "year":
            YEARS,

        "debt_gdp":
            mc02[
                "debt_gdp"
            ][
                path_index,
                :,
            ],

        "primary_balance_gdp":
            mc02[
                "primary_balance_gdp"
            ][
                path_index,
                :,
            ],

        "real_growth":
            mc02[
                "real_growth"
            ][
                path_index,
                :,
            ],

        "inflation":
            mc02[
                "inflation"
            ][
                path_index,
                :,
            ],

        "nominal_growth":
            mc02[
                "nominal_growth"
            ][
                path_index,
                :,
            ],

        "oat_30y":
            mc02[
                "oat_30y"
            ][
                path_index,
                :,
            ],

        "gg_effective_rate":
            mc02[
                "gg_effective_rate"
            ][
                path_index,
                :,
            ],
    })


REFERENCE_PATH_INDEX = 0

reference_path = (
    build_notebook02_reference_path(
        REFERENCE_PATH_INDEX
    )
)

reference_path.head()

### Timing convention

Within each year \(t\):

1. Start from previous-year stressed GG debt.
2. Compare it with previous-year debt in the Notebook 02 reference path.
3. Apply any explicit current-year fiscal, real-growth or inflation deviation.
4. Reconstruct nominal GDP growth exactly from real growth and inflation.
5. Calculate the endogenous 10Y spread feedback.
6. Transmit it to 30Y.
7. Add the 30Y feedback to the reference-path OAT.
8. Run ordinary and stressed AFT portfolios for one year using the path-specific inflation rate.
9. Measure the **incremental AFT financing cost caused only by the stress overlay**.
10. Add that incremental cost to the Notebook 02 path-specific GG effective interest rate.
11. Calculate stressed debt/GDP.
12. Feed that debt ratio into the following year's sovereign-risk equation.

This lagged structure avoids a simultaneous within-year debt/spread equation.

In [ ]:
# ============================================================
# Recursive sovereign-stress simulator
# ============================================================

def simulate_recursive_sovereign_stress_path(
    reference_path,
    *,
    primary_balance_adjustments=None,
    real_growth_adjustments=None,
    inflation_adjustments=None,
    initial_debt_gdp_adjustment=0.0,
    feedback_scale=1.0,
    transmission_lambda=LAMBDA_10Y_TO_30Y,
):
    primary_balance_adjustments = (
        {}
        if primary_balance_adjustments is None
        else dict(
            primary_balance_adjustments
        )
    )

    real_growth_adjustments = (
        {}
        if real_growth_adjustments is None
        else dict(
            real_growth_adjustments
        )
    )

    inflation_adjustments = (
        {}
        if inflation_adjustments is None
        else dict(
            inflation_adjustments
        )
    )

    base = (
        reference_path
        .set_index(
            "year"
        )
        .copy()
    )

    ordinary_portfolio = (
        portfolio.copy()
    )

    stressed_portfolio = (
        portfolio.copy()
    )

    ordinary_aft_gdp = float(
        INITIAL_GDP_EUR
    )

    stressed_aft_gdp = float(
        INITIAL_GDP_EUR
    )

    stressed_gg_gdp = float(
        INITIAL_GDP_EUR
    )

    stressed_gg_debt = float(
        INITIAL_GG_DEBT_EUR
        +
        initial_debt_gdp_adjustment
        * INITIAL_GDP_EUR
    )

    rows = []

    for i, year in enumerate(
        YEARS
    ):
        year = int(
            year
        )

        if i == 0:
            base_debt_previous = float(
                INITIAL_DEBT_GDP
            )
        else:
            base_debt_previous = float(
                base.loc[
                    YEARS[
                        i - 1
                    ],
                    "debt_gdp",
                ]
            )

        stress_debt_previous = (
            stressed_gg_debt
            / stressed_gg_gdp
        )

        pb_adjustment = float(
            primary_balance_adjustments.get(
                year,
                0.0,
            )
        )

        growth_adjustment = float(
            real_growth_adjustments.get(
                year,
                0.0,
            )
        )

        inflation_adjustment = float(
            inflation_adjustments.get(
                year,
                0.0,
            )
        )

        base_pb = float(
            base.loc[
                year,
                "primary_balance_gdp",
            ]
        )

        stress_pb = (
            base_pb
            + pb_adjustment
        )

        base_real_growth = float(
            base.loc[
                year,
                "real_growth",
            ]
        )

        base_inflation = float(
            base.loc[
                year,
                "inflation",
            ]
        )

        stress_real_growth = (
            base_real_growth
            + growth_adjustment
        )

        stress_inflation = (
            base_inflation
            + inflation_adjustment
        )

        # Exact Fisher-style composition:
        # (1 + nominal growth)
        #   = (1 + real growth) * (1 + inflation)
        base_nominal_growth = (
            (
                1
                + base_real_growth
            )
            *
            (
                1
                + base_inflation
            )
            - 1
        )

        stress_nominal_growth = (
            (
                1
                + stress_real_growth
            )
            *
            (
                1
                + stress_inflation
            )
            - 1
        )

        saved_base_nominal_growth = float(
            base.loc[
                year,
                "nominal_growth",
            ]
        )

        if not np.isclose(
            base_nominal_growth,
            saved_base_nominal_growth,
            atol=1e-12,
            rtol=1e-10,
        ):
            raise ValueError(
                f"Nominal-growth reconstruction mismatch in {year}: "
                f"reconstructed={base_nominal_growth:.12f}, "
                f"saved={saved_base_nominal_growth:.12f}"
            )

        feedback_10y = (
            calculate_10y_sovereign_feedback(
                stress_debt_gdp_previous=(
                    stress_debt_previous
                ),
                base_debt_gdp_previous=(
                    base_debt_previous
                ),
                primary_balance_adjustment=(
                    pb_adjustment
                ),
                real_growth_adjustment=(
                    growth_adjustment
                ),
                feedback_scale=(
                    feedback_scale
                ),
            )
        )

        feedback_30y = (
            transmit_feedback_to_30y(
                feedback_10y,
                transmission_lambda=(
                    transmission_lambda
                ),
            )
        )

        base_oat_30y = float(
            base.loc[
                year,
                "oat_30y",
            ]
        )

        stressed_oat_30y = (
            base_oat_30y
            + feedback_30y
        )

        ordinary_aft_summary, ordinary_portfolio = (
            simulate_portfolio_path(
                initial_portfolio=(
                    ordinary_portfolio
                ),
                initial_gdp_eur=(
                    ordinary_aft_gdp
                ),
                start_year=year,
                years=1,
                nominal_growth={
                    year:
                        base_nominal_growth
                },
                primary_balance_ratio={
                    year:
                        base_pb
                },
                market_curves={
                    year:
                        market_curves_from_30y(
                            base_oat_30y
                        )
                },
                issuance_plan=(
                    ISSUANCE_PLAN
                ),
                french_inflation={
                    year:
                        base_inflation
                },
                euro_inflation={
                    year:
                        base_inflation
                },
            )
        )

        ordinary_row = (
            ordinary_aft_summary
            .iloc[0]
        )

        ordinary_aft_gdp = float(
            ordinary_row[
                "gdp_end_eur"
            ]
        )

        stressed_aft_summary, stressed_portfolio = (
            simulate_portfolio_path(
                initial_portfolio=(
                    stressed_portfolio
                ),
                initial_gdp_eur=(
                    stressed_aft_gdp
                ),
                start_year=year,
                years=1,
                nominal_growth={
                    year:
                        stress_nominal_growth
                },
                primary_balance_ratio={
                    year:
                        stress_pb
                },
                market_curves={
                    year:
                        market_curves_from_30y(
                            stressed_oat_30y
                        )
                },
                issuance_plan=(
                    ISSUANCE_PLAN
                ),
                french_inflation={
                    year:
                        stress_inflation
                },
                euro_inflation={
                    year:
                        stress_inflation
                },
            )
        )

        stressed_row = (
            stressed_aft_summary
            .iloc[0]
        )

        stressed_aft_gdp = float(
            stressed_row[
                "gdp_end_eur"
            ]
        )

        incremental_aft_cost = (
            float(
                stressed_row[
                    "total_financing_cost_eur"
                ]
            )
            -
            float(
                ordinary_row[
                    "total_financing_cost_eur"
                ]
            )
        )

        stressed_gg_gdp_end = (
            stressed_gg_gdp
            * (
                1
                + stress_nominal_growth
            )
        )

        base_gg_effective_rate = float(
            base.loc[
                year,
                "gg_effective_rate",
            ]
        )

        ordinary_interest_on_stressed_stock = (
            base_gg_effective_rate
            * stressed_gg_debt
        )

        stressed_interest_eur = (
            ordinary_interest_on_stressed_stock
            + incremental_aft_cost
        )

        if stressed_interest_eur <= 0:
            raise ValueError(
                f"Non-positive stressed GG interest bill "
                f"in {year}: "
                f"€{stressed_interest_eur / 1e9:.2f}bn"
            )

        stressed_pb_eur = (
            stress_pb
            * stressed_gg_gdp_end
        )

        stressed_gg_debt_end = (
            stressed_gg_debt
            + stressed_interest_eur
            - stressed_pb_eur
        )

        stressed_debt_gdp_end = (
            stressed_gg_debt_end
            / stressed_gg_gdp_end
        )

        rows.append({
            "year":
                year,
            "base_debt_gdp_previous":
                base_debt_previous,
            "stress_debt_gdp_previous":
                stress_debt_previous,
            "base_primary_balance_gdp":
                base_pb,
            "stress_primary_balance_gdp":
                stress_pb,
            "primary_balance_adjustment":
                pb_adjustment,
            "base_real_growth":
                base_real_growth,
            "stress_real_growth":
                stress_real_growth,
            "real_growth_adjustment":
                growth_adjustment,
            "base_inflation":
                base_inflation,
            "stress_inflation":
                stress_inflation,
            "inflation_adjustment":
                inflation_adjustment,
            "base_nominal_growth":
                base_nominal_growth,
            "stress_nominal_growth":
                stress_nominal_growth,
            "feedback_10y":
                feedback_10y,
            "feedback_30y":
                feedback_30y,
            "base_oat_30y":
                base_oat_30y,
            "stress_oat_30y":
                stressed_oat_30y,
            "incremental_aft_cost_eur":
                incremental_aft_cost,
            "base_gg_effective_rate":
                base_gg_effective_rate,
            "stress_interest_gdp":
                stressed_interest_eur
                / stressed_gg_gdp_end,
            "base_debt_gdp_end":
                float(
                    base.loc[
                        year,
                        "debt_gdp",
                    ]
                ),
            "stress_debt_gdp_end":
                stressed_debt_gdp_end,
        })

        stressed_gg_gdp = (
            stressed_gg_gdp_end
        )

        stressed_gg_debt = (
            stressed_gg_debt_end
        )

    return pd.DataFrame(
        rows
    )

In [ ]:
# ============================================================
# Exact Notebook 02 reproduction test
# ============================================================

identity_path = (
    simulate_recursive_sovereign_stress_path(
        reference_path=reference_path,
        initial_debt_gdp_adjustment=0.0,
        feedback_scale=1.0,
    )
)

debt_error = (
    identity_path[
        "stress_debt_gdp_end"
    ].to_numpy()
    -
    reference_path[
        "debt_gdp"
    ].to_numpy()
)

oat_error = (
    identity_path[
        "stress_oat_30y"
    ].to_numpy()
    -
    reference_path[
        "oat_30y"
    ].to_numpy()
)

print(
    "Maximum debt/GDP error:",
    np.max(
        np.abs(
            debt_error
        )
    ),
)

print(
    "Maximum OAT error:",
    np.max(
        np.abs(
            oat_error
        )
    ),
)

assert np.allclose(
    identity_path[
        "stress_debt_gdp_end"
    ],
    reference_path[
        "debt_gdp"
    ],
    atol=1e-10,
)

assert np.allclose(
    identity_path[
        "stress_oat_30y"
    ],
    reference_path[
        "oat_30y"
    ],
    atol=1e-12,
)

assert np.allclose(
    identity_path[
        "feedback_10y"
    ],
    0.0,
    atol=1e-12,
)

assert np.allclose(
    identity_path[
        "incremental_aft_cost_eur"
    ],
    0.0,
    atol=1e-4,
)

print(
    "Exact Notebook 02 reproduction test passed."
)

## 10. +10 pp initial-debt impulse response

This is a **unit test of the feedback loop**, not a forecast or scenario probability.

The stressed economy starts 2027 with debt/GDP 10 percentage points above the Notebook 02 reference path. No fiscal or growth assumptions are otherwise changed.

The experiment asks:

> If France entered the horizon with an additional 10 pp of debt, how much extra 10Y/30Y sovereign spread, AFT financing cost and subsequent debt would the calibrated feedback mechanism generate?

In [ ]:
# ============================================================
# +10 pp debt impulse
# ============================================================

debt_impulse_path = (
    simulate_recursive_sovereign_stress_path(
        reference_path=reference_path,
        initial_debt_gdp_adjustment=0.10,
        feedback_scale=1.0,
    )
)

impulse_display = (
    debt_impulse_path[
        [
            "year",
            "base_debt_gdp_end",
            "stress_debt_gdp_end",
            "feedback_10y",
            "feedback_30y",
            "base_oat_30y",
            "stress_oat_30y",
            "incremental_aft_cost_eur",
            "stress_interest_gdp",
        ]
    ]
    .copy()
)

for column in [
    "base_debt_gdp_end",
    "stress_debt_gdp_end",
    "feedback_10y",
    "feedback_30y",
    "base_oat_30y",
    "stress_oat_30y",
    "stress_interest_gdp",
]:
    impulse_display[
        column
    ] *= 100

impulse_display[
    "incremental_aft_cost_bn"
] = (
    impulse_display[
        "incremental_aft_cost_eur"
    ]
    / 1e9
)

impulse_display = (
    impulse_display
    .drop(
        columns=[
            "incremental_aft_cost_eur",
        ]
    )
)

impulse_display.round(3)

In [ ]:
# ============================================================
# Recursive debt diagnostic
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.plot(
    debt_impulse_path[
        "year"
    ],
    debt_impulse_path[
        "base_debt_gdp_end"
    ]
    * 100,
    label="Notebook 02 reference path",
)

ax.plot(
    debt_impulse_path[
        "year"
    ],
    debt_impulse_path[
        "stress_debt_gdp_end"
    ]
    * 100,
    label="+10 pp initial debt + endogenous feedback",
)

ax.set_title(
    "Debt impulse response with endogenous sovereign feedback"
)

ax.set_xlabel("Year")
ax.set_ylabel("Debt / GDP (%)")
ax.grid(alpha=0.25)
ax.legend()

plt.show()

In [ ]:
# ============================================================
# Recursive spread diagnostic
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.plot(
    debt_impulse_path[
        "year"
    ],
    debt_impulse_path[
        "feedback_10y"
    ]
    * 10_000,
    label="Additional 10Y spread",
)

ax.plot(
    debt_impulse_path[
        "year"
    ],
    debt_impulse_path[
        "feedback_30y"
    ]
    * 10_000,
    label="Additional 30Y spread",
)

ax.axhline(
    0,
    linewidth=0.8,
)

ax.set_title(
    "Endogenous sovereign-spread impulse response"
)

ax.set_xlabel("Year")
ax.set_ylabel("Basis points")
ax.grid(alpha=0.25)
ax.legend()

plt.show()

# Recursive-engine validation status

The recursive engine now uses the actual path-specific Notebook 02:

- real GDP growth;
- inflation;
- nominal GDP growth;
- primary balance;
- 30Y OAT;
- general-government effective financing rate.

No hard-coded inflation path remains.

The zero-stress identity test therefore checks the strongest possible condition:

\[
\boxed{\text{Notebook 03 with zero stress}=\text{Notebook 02}}
\]

The following fiscal-consolidation scenarios can now be interpreted as genuine overlays on the validated Notebook 02 Monte Carlo paths.

# 11. Fiscal-consolidation comparison

The recursive engine has passed the debt-impulse decomposition test.

We now move from a unit test to an economic scenario comparison:

- **Full consolidation** — retain the Notebook 02 fiscal path.
- **Half consolidation** — deliver only 50% of the structural improvement from the 2026 starting primary balance to the central fiscal path.
- **No consolidation** — keep the structural primary balance at its 2026 starting level, while preserving the ordinary-times stochastic deviation already present in the selected Notebook 02 path.

The scenario adjustment is therefore applied **around** the Notebook 02 path rather than replacing its stochastic component.

In [ ]:
# ============================================================
# Select a representative Notebook 02 Monte Carlo path
# ============================================================

REPRESENTATIVE_VARIABLES = [
    "debt_gdp",
    "oat_30y",
    "primary_balance_gdp",
    "nominal_growth",
]


def standardized_distance_to_median(
    values,
):
    median_path = np.median(
        values,
        axis=0,
    )

    scale = np.std(
        values,
        axis=0,
    )

    scale = np.where(
        scale > 0,
        scale,
        1.0,
    )

    return np.mean(
        (
            (
                values
                - median_path
            )
            / scale
        )
        ** 2,
        axis=1,
    )


representative_distance = np.zeros(
    mc02[
        "debt_gdp"
    ].shape[0]
)

for variable in REPRESENTATIVE_VARIABLES:
    representative_distance += (
        standardized_distance_to_median(
            mc02[
                variable
            ]
        )
    )


REPRESENTATIVE_PATH_INDEX = int(
    np.argmin(
        representative_distance
    )
)


reference_path = (
    build_notebook02_reference_path(
        REPRESENTATIVE_PATH_INDEX
    )
)


print(
    "Representative Notebook 02 path index:",
    REPRESENTATIVE_PATH_INDEX,
)

print(
    "Combined standardized distance:",
    round(
        representative_distance[
            REPRESENTATIVE_PATH_INDEX
        ],
        4,
    ),
)


reference_path

In [ ]:
# ============================================================
# Revalidate exact identity on the representative path
# ============================================================

representative_identity_path = (
    simulate_recursive_sovereign_stress_path(
        reference_path=reference_path,
        feedback_scale=1.0,
    )
)

assert np.allclose(
    representative_identity_path[
        "stress_debt_gdp_end"
    ],
    reference_path[
        "debt_gdp"
    ],
    atol=1e-10,
)

assert np.allclose(
    representative_identity_path[
        "stress_oat_30y"
    ],
    reference_path[
        "oat_30y"
    ],
    atol=1e-12,
)

assert np.allclose(
    representative_identity_path[
        "feedback_10y"
    ],
    0.0,
    atol=1e-12,
)

print(
    "Representative-path identity test passed."
)

In [ ]:
# ============================================================
# Check representative path against Monte Carlo medians
# ============================================================

representative_check = pd.DataFrame({
    "year":
        YEARS,

    "reference_debt_gdp":
        reference_path[
            "debt_gdp"
        ].to_numpy(),

    "median_debt_gdp":
        np.median(
            mc02[
                "debt_gdp"
            ],
            axis=0,
        ),

    "reference_oat_30y":
        reference_path[
            "oat_30y"
        ].to_numpy(),

    "median_oat_30y":
        np.median(
            mc02[
                "oat_30y"
            ],
            axis=0,
        ),

    "reference_primary_balance":
        reference_path[
            "primary_balance_gdp"
        ].to_numpy(),

    "median_primary_balance":
        np.median(
            mc02[
                "primary_balance_gdp"
            ],
            axis=0,
        ),
})


for column in [
    "reference_debt_gdp",
    "median_debt_gdp",
    "reference_oat_30y",
    "median_oat_30y",
    "reference_primary_balance",
    "median_primary_balance",
]:
    representative_check[
        column
    ] *= 100


representative_check.round(
    2
)

## 12. Structural fiscal-adjustment paths

The 2026 primary balance starts at:

\[
PB_{2026}=-2.7\%\text{ of GDP}.
\]

The central structural path improves to approximately zero by 2031 and is held there afterwards.

For a consolidation fraction \(c\):

\[
PB^{scenario}_t
=
PB^{reference}_t
+
(1-c)
\left(
PB_{2026}
-
PB^{central}_t
\right).
\]

Hence:

- \(c=1\): no adjustment to the reference path;
- \(c=0.5\): only half the central structural consolidation is achieved;
- \(c=0\): no structural consolidation beyond the 2026 starting position.

In [ ]:
# ============================================================
# Central structural primary-balance path
# ============================================================

STARTING_PRIMARY_BALANCE = -0.027

CENTRAL_PRIMARY_BALANCE = {
    2027: -0.023,
    2028: -0.017,
    2029: -0.011,
    2030: -0.006,
    2031:  0.000,
}

for year in range(
    2032,
    max(YEARS) + 1,
):
    CENTRAL_PRIMARY_BALANCE[
        year
    ] = 0.000


def build_fiscal_adjustments(
    reference_path,
    consolidation_fraction,
):
    """
    Return year -> PB adjustment relative to the Notebook 02
    reference path.

    The stochastic deviation in the reference path is preserved;
    only the structural consolidation component is weakened.
    """

    if not (
        0.0
        <= consolidation_fraction
        <= 1.0
    ):
        raise ValueError(
            "consolidation_fraction must lie between 0 and 1."
        )

    adjustments = {}

    for year in (
        reference_path[
            "year"
        ]
        .astype(int)
    ):
        central_pb = float(
            CENTRAL_PRIMARY_BALANCE[
                year
            ]
        )

        structural_shortfall = (
            STARTING_PRIMARY_BALANCE
            - central_pb
        )

        adjustments[
            year
        ] = (
            1.0
            - consolidation_fraction
        ) * structural_shortfall

    return adjustments

In [ ]:
# ============================================================
# Build scenario adjustments
# ============================================================

CONSOLIDATION_SCENARIOS = {
    "Full consolidation":
        1.0,

    "Half consolidation":
        0.5,

    "No consolidation":
        0.0,
}


fiscal_adjustments = {
    scenario:
        build_fiscal_adjustments(
            reference_path=reference_path,
            consolidation_fraction=fraction,
        )
    for scenario, fraction
    in CONSOLIDATION_SCENARIOS.items()
}


fiscal_path_preview = pd.DataFrame({
    "year":
        YEARS,

    "reference_pb":
        reference_path[
            "primary_balance_gdp"
        ].to_numpy(),
})


for scenario in (
    CONSOLIDATION_SCENARIOS
):
    fiscal_path_preview[
        scenario
    ] = [
        (
            reference_path
            .set_index(
                "year"
            )
            .loc[
                year,
                "primary_balance_gdp",
            ]
            +
            fiscal_adjustments[
                scenario
            ][
                year
            ]
        )
        for year in YEARS
    ]


display_fiscal_path = (
    fiscal_path_preview.copy()
)

for column in (
    display_fiscal_path.columns
):
    if column != "year":
        display_fiscal_path[
            column
        ] *= 100


display_fiscal_path.round(
    2
)

## 13. Run the three recursive fiscal scenarios

Each scenario now feeds through:

\[
PB_t
\rightarrow
Spread^{10Y}_t
\rightarrow
Spread^{30Y}_t
\rightarrow
OAT^{30Y}_t
\rightarrow
AFT\ refinancing
\rightarrow
GG\ interest
\rightarrow
Debt/GDP
\]

and the new debt ratio becomes the input to the following year's sovereign-risk equation.

In [ ]:
# ============================================================
# Run recursive fiscal scenarios
# ============================================================

scenario_paths = {}


for scenario in (
    CONSOLIDATION_SCENARIOS
):
    print(
        f"Running: {scenario}"
    )

    scenario_paths[
        scenario
    ] = (
        simulate_recursive_sovereign_stress_path(
            reference_path=reference_path,
            primary_balance_adjustments=(
                fiscal_adjustments[
                    scenario
                ]
            ),
            feedback_scale=1.0,
        )
    )


print(
    "All fiscal scenarios completed."
)

In [ ]:
# ============================================================
# Scenario summary
# ============================================================

summary_rows = []


for scenario, path in (
    scenario_paths.items()
):
    summary_rows.append({
        "scenario":
            scenario,

        "terminal_debt_gdp_pct":
            (
                path[
                    "stress_debt_gdp_end"
                ]
                .iloc[
                    -1
                ]
                * 100
            ),

        "peak_debt_gdp_pct":
            (
                path[
                    "stress_debt_gdp_end"
                ]
                .max()
                * 100
            ),

        "terminal_oat_30y_pct":
            (
                path[
                    "stress_oat_30y"
                ]
                .iloc[
                    -1
                ]
                * 100
            ),

        "peak_oat_30y_pct":
            (
                path[
                    "stress_oat_30y"
                ]
                .max()
                * 100
            ),

        "terminal_interest_gdp_pct":
            (
                path[
                    "stress_interest_gdp"
                ]
                .iloc[
                    -1
                ]
                * 100
            ),

        "peak_interest_gdp_pct":
            (
                path[
                    "stress_interest_gdp"
                ]
                .max()
                * 100
            ),

        "terminal_10y_feedback_bp":
            (
                path[
                    "feedback_10y"
                ]
                .iloc[
                    -1
                ]
                * 10_000
            ),

        "terminal_30y_feedback_bp":
            (
                path[
                    "feedback_30y"
                ]
                .iloc[
                    -1
                ]
                * 10_000
            ),

        "cumulative_incremental_aft_cost_bn":
            (
                path[
                    "incremental_aft_cost_eur"
                ]
                .sum()
                / 1e9
            ),
    })


scenario_summary = (
    pd.DataFrame(
        summary_rows
    )
    .set_index(
        "scenario"
    )
)


scenario_summary.round(
    2
)

In [ ]:
# ============================================================
# Debt/GDP comparison
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)


for scenario, path in (
    scenario_paths.items()
):
    ax.plot(
        path[
            "year"
        ],
        path[
            "stress_debt_gdp_end"
        ]
        * 100,
        label=scenario,
    )


ax.set_title(
    "France — Fiscal consolidation and recursive debt dynamics"
)

ax.set_xlabel(
    "Year"
)

ax.set_ylabel(
    "Debt / GDP (%)"
)

ax.grid(
    alpha=0.25
)

ax.legend()

plt.show()

In [ ]:
# ============================================================
# Primary-balance comparison
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)


for scenario, path in (
    scenario_paths.items()
):
    ax.plot(
        path[
            "year"
        ],
        path[
            "stress_primary_balance_gdp"
        ]
        * 100,
        label=scenario,
    )


ax.axhline(
    0,
    linewidth=0.8,
)


ax.set_title(
    "France — Primary balance under consolidation scenarios"
)

ax.set_xlabel(
    "Year"
)

ax.set_ylabel(
    "Primary balance / GDP (%)"
)

ax.grid(
    alpha=0.25
)

ax.legend()

plt.show()

In [ ]:
# ============================================================
# 30Y OAT comparison
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)


for scenario, path in (
    scenario_paths.items()
):
    plot_years = np.concatenate(
        [
            [
                CURRENT_YEAR
            ],
            path[
                "year"
            ].to_numpy(),
        ]
    )

    plot_oat = np.concatenate(
        [
            [
                CURRENT_OAT_30Y
            ],
            path[
                "stress_oat_30y"
            ].to_numpy(),
        ]
    )

    ax.plot(
        plot_years,
        plot_oat * 100,
        label=scenario,
    )


ax.scatter(
    [
        CURRENT_YEAR
    ],
    [
        CURRENT_OAT_30Y
        * 100
    ],
    zorder=5,
    label="2026 observed",
)


ax.set_title(
    "France — 30Y OAT under fiscal-consolidation scenarios"
)

ax.set_xlabel(
    "Year"
)

ax.set_ylabel(
    "30Y OAT yield (%)"
)

ax.grid(
    alpha=0.25
)

ax.legend()

plt.show()

In [ ]:
# ============================================================
# Endogenous 10Y sovereign feedback
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)


for scenario, path in (
    scenario_paths.items()
):
    ax.plot(
        path[
            "year"
        ],
        path[
            "feedback_10y"
        ]
        * 10_000,
        label=scenario,
    )


ax.axhline(
    0,
    linewidth=0.8,
)


ax.set_title(
    "France — Endogenous 10Y sovereign-spread feedback"
)

ax.set_xlabel(
    "Year"
)

ax.set_ylabel(
    "Additional spread (bp)"
)

ax.grid(
    alpha=0.25
)

ax.legend()

plt.show()

In [ ]:
# ============================================================
# Interest burden comparison
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 6)
)


for scenario, path in (
    scenario_paths.items()
):
    ax.plot(
        path[
            "year"
        ],
        path[
            "stress_interest_gdp"
        ]
        * 100,
        label=scenario,
    )


ax.set_title(
    "France — General-government interest burden"
)

ax.set_xlabel(
    "Year"
)

ax.set_ylabel(
    "Interest / GDP (%)"
)

ax.grid(
    alpha=0.25
)

ax.legend()

plt.show()

## 14. Decompose the no-consolidation scenario

To isolate the contribution of the sovereign-risk loop, rerun the no-consolidation fiscal path with the spread feedback switched off.

The difference between the two no-consolidation paths measures the additional debt generated specifically by:

\[
Fiscal\ slippage
\rightarrow
Spread
\rightarrow
Refinancing\ cost
\rightarrow
Debt.
\]

In [ ]:
# ============================================================
# No-consolidation: with vs without sovereign feedback
# ============================================================

no_consolidation_without_feedback = (
    simulate_recursive_sovereign_stress_path(
        reference_path=reference_path,
        primary_balance_adjustments=(
            fiscal_adjustments[
                "No consolidation"
            ]
        ),
        feedback_scale=0.0,
    )
)


no_consolidation_with_feedback = (
    scenario_paths[
        "No consolidation"
    ]
)


no_consolidation_decomposition = pd.DataFrame({
    "year":
        YEARS,

    "reference_debt_gdp":
        reference_path[
            "debt_gdp"
        ].to_numpy(),

    "no_consolidation_no_feedback":
        no_consolidation_without_feedback[
            "stress_debt_gdp_end"
        ].to_numpy(),

    "no_consolidation_with_feedback":
        no_consolidation_with_feedback[
            "stress_debt_gdp_end"
        ].to_numpy(),
})


no_consolidation_decomposition[
    "fiscal_mechanical_effect_pp"
] = (
    (
        no_consolidation_decomposition[
            "no_consolidation_no_feedback"
        ]
        -
        no_consolidation_decomposition[
            "reference_debt_gdp"
        ]
    )
    * 100
)


no_consolidation_decomposition[
    "sovereign_feedback_effect_pp"
] = (
    (
        no_consolidation_decomposition[
            "no_consolidation_with_feedback"
        ]
        -
        no_consolidation_decomposition[
            "no_consolidation_no_feedback"
        ]
    )
    * 100
)


no_consolidation_decomposition[
    "total_effect_pp"
] = (
    (
        no_consolidation_decomposition[
            "no_consolidation_with_feedback"
        ]
        -
        no_consolidation_decomposition[
            "reference_debt_gdp"
        ]
    )
    * 100
)


no_consolidation_decomposition.round(
    3
)

In [ ]:
# ============================================================
# Save scenario outputs
# ============================================================

scenario_summary.to_csv(
    STRESS_DIR
    / "fiscal_consolidation_scenario_summary.csv"
)


no_consolidation_decomposition.to_csv(
    STRESS_DIR
    / "no_consolidation_decomposition.csv",
    index=False,
)


for scenario, path in (
    scenario_paths.items()
):
    safe_name = (
        scenario
        .lower()
        .replace(
            " ",
            "_",
        )
    )

    path.to_csv(
        STRESS_DIR
        / f"{safe_name}_recursive_path.csv",
        index=False,
    )


print(
    "Saved fiscal-consolidation scenario outputs to:",
    STRESS_DIR,
)

# Next step

The single-path recursive engine is now internally consistent with the regenerated Notebook 02 outputs.

The next model stage is the **full sovereign-stress Monte Carlo**:

\[
10{,}000\ \text{Notebook 02 paths}
\rightarrow
\text{recursive sovereign feedback}
\rightarrow
\text{full / half / no-consolidation distributions}.
\]

This will let us compare conditional probabilities such as:

\[
P(Debt/GDP_{2041}>150\%)
\]

\[
P(OAT_{30Y}>7\%)
\]

and

\[
P(Interest/GDP>5\%)
\]

under each fiscal-consolidation regime.